# **CardiAg: Raincloud Plots**

Pipeline for creating the raincloud plots featured in the CardiAg manuscript. 

Author: Marta Gerosa

Created on: 22 April 2026

## **Pipeline structure**

1.  **Cardiac phase analysis - raincloud plots [Results 2.2.3 & 2.5.1]**: import the datasets containing the intentional binding averages according to cardiac phase, namely `task-CardiAgIBTask_ecg_actbinding_avg.tsv` and `task-CardiAgIBTask_ecg_tonebinding_avg.tsv` for action and tone binding, respectively, which were generated during the `CardiAg_cardiac_phase_analysis` pipeline and stored in `derivatives\cardiac-phase-analysis`. These are used to produce raincloud plots of intentional binding measures, according to cardiac phase. These are reported in **Results section 2.2.3**, with corresponding **Figures 3c-d**. Furthermore, import the datasets containing the averages of the pre- and post-action R-R interval changes, namely `task-CardiAgIBTask_preact_cardiac_deceleration.tsv` and `task-CardiAgIBTask_postact_cardiac_acceleration.tsv`, which were generated during the `CardiAg_cardiac_phase_analysis` pipeline and stored in `derivatives\cardiac-phase-analysis`. These are used to produce raincloud plots of pre-action cardiac deceleration and post-action cardiac acceleration slopes (z-scored), according to intentional binding task conditions. These are reported in **Results section 2.5.1**, with corresponding **Figures 6b-c**. 
2.  **Respiratory phase analysis - raincloud plots [Results 2.3.3]**: import the datasets containing the intentional binding averages according to respiratory phase, namely `task-CardiAgIBTask_resp_actbinding_avg.tsv` and `task-CardiAgIBTask_resp_tonebinding_avg.tsv` for action and tone binding, respectively, which were generated during the `CardiAg_resp_phase_analysis` pipeline and stored in `derivatives\resp-phase-analysis`. These are used to produce raincloud plots of intentional binding measures, according to respiratory phase. These are reported in **Results section 2.3.3**, with corresponding **Figures 4c-d**.
3.  **Cardiorespiratory analysis - raincloud plots [Results 2.4]**: import the datasets containing the intentional binding averages according to cardio-respiratory phase, namely `task-CardiAgIBTask_ecg_resp_actbinding_avg.tsv` and `task-CardiAgIBTask_ecg_resp_tonebinding_avg.tsv` for action and tone binding, respectively, which were generated during the `CardiAg_cardioresp_phase_analysis` pipeline and stored in `derivatives\cardioresp-phase-analysis`. These are used to produce raincloud plots of intentional binding measures, according to cardiac and respiratory phase. These are reported in **Results section 2.4**, with corresponding **Figures 5a-b**.

In [1]:
############## Import modules ##############

import pandas as pd
import os
import matplotlib.pyplot as plt
import ptitprince as pt
import seaborn as sns

In [ ]:
############## Specify directories of data input and output ##############

# Specify the directory of derivatives
wd = r'.\data' # directory of data storage
exp_name = 'CardiAgIBTask' 
datatype_name = 'beh'           # current datatype folder according to BIDS

# Create directory for saving plots
project_root = os.path.dirname(wd)
plotting_dir = os.path.join(project_root, 'results', 'datavisualization')
if not os.path.exists(plotting_dir):
    os.makedirs(plotting_dir)

# Define function to load the TSV file for plotting
def load_dataset(wd, folder_name, dataset_name):

    dir = os.path.join(wd, 'derivatives', folder_name, dataset_name)
    dataset = pd.read_csv(dir, sep='\t', header=0)

    return dataset

## **1. Cardiac phase analysis - raincloud plots [Results 2.2.3 & 2.5.1]**

This section creates raincloud plots related to cardiac phase analysis. In detail: 

- **1a. Cardiac phase binary analysis [Results 2.2.3]**: import the datasets containing the intentional binding averages according to cardiac phase, namely `task-CardiAgIBTask_ecg_actbinding_avg.tsv` and `task-CardiAgIBTask_ecg_tonebinding_avg.tsv` for action and tone binding, respectively, which were generated during the `CardiAg_cardiac_phase_analysis` pipeline and stored in `derivatives\cardiac-phase-analysis`. These are used to produce raincloud plots of intentional binding measures, according to cardiac phase. These are reported in **Results section 2.2.3**, with corresponding **Figures 3c-d**. 
- **1b. Cardiac phase - R-R interval changes [Results 2.5.1]**: import the datasets containing the averages of the pre- and post-action R-R interval changes, namely `task-CardiAgIBTask_preact_cardiac_deceleration.tsv` and `task-CardiAgIBTask_postact_cardiac_acceleration.tsv`, which were generated during the `CardiAg_cardiac_phase_analysis` pipeline and stored in `derivatives\cardiac-phase-analysis`. These are used to produce raincloud plots of pre-action cardiac deceleration and post-action cardiac acceleration slopes (z-scored), according to intentional binding task conditions. These are reported in **Results section 2.5.1**, with corresponding **Figures 6b-c**. 

### **1a. Cardiac phase binary analysis [Results 2.2.3]**

In [ ]:
############## 1a. Cardiac phase binary analysis [Results 2.2.3] ##############

# Define function to create plots of intentional binding binned in systole vs. diastole
def plot_sysdia_binding(df_means_long, xy_colnames, xlabel, ylabel, ylim, svg_fname):

    # Set custom colors for the sys/dia binning
    colors = ['#ff7f00', '#7AC5CD']

    # Plot boxplots with individual datapoints
    plt.figure(figsize=(5,5), dpi=300)

    plt.axhline(y=0, linewidth=0.6, alpha=0.4, color='k', linestyle='--') # horizontal zero line

    # Add raincloud plots divided by cardiac phase
    pt.RainCloud(data=df_means_long, x=xy_colnames[0], y=xy_colnames[1], palette=colors, 
                 alpha=0.8, bw=0.3, width_viol=0.4, width_box=0.08, 
                 point_size=3, orient='v', linewidth=1.2, 
                 box_whiskerprops = {'linewidth': 1.2, "zorder": 10})
    
    # Add labels and formatting
    plt.xlabel(xlabel, fontsize='x-large')
    plt.ylabel(ylabel, fontsize='x-large')
    plt.ylim(ylim[0], ylim[1])
    plt.xticks(ticks=[0, 1], labels=['Systole', 'Diastole'], fontsize='large')
    sns.despine()

    # Save plot 
    fig = os.path.join(plotting_dir, svg_fname)
    plt.tight_layout()
    plt.savefig(fig, format="svg")
    
    return plt.show()


# Import dataset with action binding means in systole vs. diastole
actbind_sysdia = load_dataset(wd, 'cardiac-phase-analysis', 
                              f'task-{exp_name}_ecg_actbinding_avg.tsv')

### FIGURE 3c ### 
# Raincloud plot of action binding according to task event in sys/dia
plot_sysdia_binding(df_means_long=actbind_sysdia, xy_colnames=('phase', 'act_binding'),
                    xlabel='Cardiac phase at action onset',
                    ylabel='Action binding: OpA - BasA (ms)', 
                    ylim=(-200,400), svg_fname='Figure3c_cardiac_phase_actbinding.svg')



# Import dataset with tone binding means in systole vs. diastole
tonebind_sysdia = load_dataset(wd, 'cardiac-phase-analysis', 
                               f'task-{exp_name}_ecg_tonebinding_avg.tsv')

### FIGURE 3d ### 
# Raincloud plot of tone binding according to task event in sys/dia
plot_sysdia_binding(df_means_long=tonebind_sysdia, xy_colnames=('phase', 'tone_binding'), 
                    xlabel='Cardiac phase at tone onset',
                    ylabel='Tone binding: OpT - BasT (ms)', 
                    ylim=(-400,200), svg_fname='Figure3d_cardiac_phase_tonebinding.svg')

### **1b. Cardiac phase - R-R interval changes [Results 2.5.1]**

In [ ]:
############## 1b. Cardiac phase - R-R interval changes [Results 2.5.1] ##############

# Import dataset with pre-action cardiac deceleration
card_decel = load_dataset(wd, 'cardiac-phase-analysis', 
                          f'task-{exp_name}_preact_cardiac_deceleration.tsv')

# Define color palette by IB task condition
#cond_colpalette = {'BasA': '#a4cbb6', 'OpA': '#007a4e', 'OpT': '#0b3142'} 
cond_colpalette = ['#a4cbb6', '#007a4e', '#0b3142']

### FIGURE 6b ###
# Raincloud plot of pre-action cardiac deceleration slopes by IB task condition
plt.figure(figsize=(6.5, 6.5), dpi=300)
plt.axvline(x=0, linewidth=0.6, alpha=0.4, color='k', linestyle='--')

# Rainclouds plot according to condition
pt.RainCloud(data=card_decel, x='Condition', y='Deceleration', palette=cond_colpalette, 
             alpha=0.8, bw=0.3, width_viol=0.8, width_box=0.08, 
             point_size=3, orient='h', linewidth=1, 
             box_whiskerprops = {'linewidth': 1, "zorder": 10}, box_linewidth=1)

plt.xlabel('Pre-action cardiac deceleration T-1 to T0 (z-scores)', fontsize='x-large')
plt.ylabel('Condition', fontsize='x-large')
plt.xlim(-0.8, 0.8)
sns.despine()

# Save plot 
fig6b = os.path.join(plotting_dir, 'Figure6b_preact_cardiac_deceleration.svg')
plt.tight_layout()
plt.savefig(fig6b, format="svg")
plt.show()


# Import dataset with post-action cardiac acceleration
card_accel_long = load_dataset(wd,'cardiac-phase-analysis', 
                               f'task-{exp_name}_postact_cardiac_acceleration.tsv')

### FIGURE 6c ###
# Raincloud plot of post-action cardiac acceleration slopes by IB task condition
plt.figure(figsize=(6.5, 6.5), dpi=300)
plt.axvline(x=0, linewidth=0.6, alpha=0.4, color='k', linestyle='--')

# Rainclouds plot according to condition
pt.RainCloud(data=card_accel_long, x='Condition', y='Acceleration', palette=cond_colpalette, 
             alpha=0.8, bw=0.3, width_viol=0.8, width_box=0.08, point_size=3, orient='h', linewidth=1, 
             box_whiskerprops = {'linewidth': 1, "zorder": 10}, box_linewidth=1)

sns.despine()

plt.xlabel('Post-action cardiac acceleration T0 to T+1 (z-scores)', fontsize='x-large')
plt.ylabel('Condition', fontsize='x-large')
plt.xlim(-0.8, 0.8)

# Save plot 
fig6c = os.path.join(plotting_dir, 'Figure6c_postact_cardiac_acceleration.svg')
plt.savefig(fig6c, format="svg")
plt.show()

## **2. Respiratory phase analysis - raincloud plots [Results 2.3.3]**

This section imports the datasets containing the intentional binding averages according to respiratory phase, namely `task-CardiAgIBTask_resp_actbinding_avg.tsv` and `task-CardiAgIBTask_resp_tonebinding_avg.tsv` for action and tone binding, respectively, which were generated during the `CardiAg_resp_phase_analysis` pipeline and stored in `derivatives\resp-phase-analysis`. These are used to produce raincloud plots of intentional binding measures, according to respiratory phase. These are reported in **Results section 2.3.3**, with corresponding **Figures 4c-d**.

In [ ]:
############## 2. Respiratory phase analysis - raincloud plots [Results 2.3.3] ##############

# Define function to create plots of intentional binding binned in expiration vs. inspiration
def plot_expinsp_binding(df_means_long, xy_colnames, xlabel, ylabel, ylim, svg_fname):

    # Set custom colors for the exp/ins binning
    colors = ['#8E44AD', '#8BBB94']

    # Plot boxplots with individual datapoints
    plt.figure(figsize=(5,5), dpi=300)

    plt.axhline(y=0, linewidth=0.6, alpha=0.4, color='k', linestyle='--') # horizontal zero line

    # Add raincloud plots divided by respiratory phase
    pt.RainCloud(data=df_means_long, x=xy_colnames[0], y=xy_colnames[1], palette=colors, 
                 alpha=0.8, bw=0.3, width_viol=0.4, width_box=0.08, 
                 point_size=3, orient='v', linewidth=1.2, 
                 box_whiskerprops = {'linewidth': 1.2, "zorder": 10})
    
    # Add labels and formatting
    plt.xlabel(xlabel, fontsize='x-large')
    plt.ylabel(ylabel, fontsize='x-large')
    plt.ylim(ylim[0], ylim[1])
    plt.xticks(ticks=[0, 1], labels=['Expiration', 'Inspiration'], fontsize='large')
    sns.despine()

    # Save plot 
    fig = os.path.join(plotting_dir, svg_fname)
    plt.tight_layout()
    plt.savefig(fig, format="svg")

    return plt.show()

# Import dataset with action binding means in expiration vs. inspiration
actbind_expinsp = load_dataset(wd, 'resp-phase-analysis', 
                               f'task-{exp_name}_resp_actbinding_avg.tsv')

### FIGURE 4c ### 
# Raincloud plot of action binding according to task event in exp/ins
plot_expinsp_binding(df_means_long=actbind_expinsp, xy_colnames=('phase', 'act_binding'),
                     xlabel='Respiratory phase at action onset',
                     ylabel='Action binding: OpA - BasA (ms)', 
                     ylim=(-200,400), svg_fname='Figure4c_resp_phase_actbinding.svg')


# Import dataset with tone binding means in expiration vs. inspiration
tonebind_expinsp = load_dataset(wd, 'resp-phase-analysis', 
                               f'task-{exp_name}_resp_tonebinding_avg.tsv')

### FIGURE 4d ### 
# Raincloud plot of tone binding according to task event in exp/ins
plot_expinsp_binding(df_means_long=tonebind_expinsp, xy_colnames=('phase', 'tone_binding'), 
                     xlabel='Respiratory phase at tone onset',
                     ylabel='Tone binding: OpT - BasT (ms)', 
                     ylim=(-400,200), svg_fname='Figure4d_resp_phase_tonebinding.svg')

## **3. Cardiorespiratory analysis - raincloud plots [Results 2.4]**

This section imports the datasets containing the intentional binding averages according to cardio-respiratory phase, namely `task-CardiAgIBTask_ecg_resp_actbinding_avg.tsv` and `task-CardiAgIBTask_ecg_resp_tonebinding_avg.tsv` for action and tone binding, respectively, which were generated during the `CardiAg_cardioresp_phase_analysis` pipeline and stored in `derivatives\cardioresp-phase-analysis`. These are used to produce raincloud plots of intentional binding measures, according to cardiac and respiratory phase. These are reported in **Results section 2.4**, with corresponding **Figures 5a-b**.

In [ ]:
############## 3. Cardiorespiratory analysis - raincloud plots [Results 2.4] ##############

# Define function to plot interaction of cardiac and respiratory phase on intentional binding
def plot_cardioresp_binding(df_means_long, yvar, ylabel, xlabel, ylim, palette, svg_fname):
    
    # Divide plots by cardiac phase
    g = sns.FacetGrid(df_means_long, col="cardiac",
                      height=5, aspect=0.7, sharey=True)

    g.fig.set_dpi(300)

    for ax in g.axes.flat:
        ax.axhline(0, linestyle="--", linewidth=0.6, alpha=0.5, color='k')
        ax.set_xlabel(xlabel)
        ax.set_ylim(ylim)
        ax.tick_params(axis='both', labelsize=16)

    # Raincloud plots according to respiratory phase
    def rain(data, **kwargs):
        pt.RainCloud(data=data, x="resp", y=yvar,
                     palette=palette,
                     width_viol=0.5, width_box=0.15, point_size=3, bw=0.3, orient="v")

    g.map_dataframe(rain)

    g.set_axis_labels("", "")
    g.set_titles("{col_name}", size=16)
    g.fig.supylabel(ylabel, fontsize=16)
    g.fig.supxlabel(xlabel, fontsize=16)

    fig = os.path.join(plotting_dir, svg_fname)
    plt.tight_layout()
    plt.savefig(fig, format="svg")

    plt.show()


# Import dataset with action binding means according to cardio-respiratory phase
actbind_cardioresp = load_dataset(wd, 'cardioresp-phase-analysis', 
                                  f'task-{exp_name}_ecg_resp_actbinding_avg.tsv')

# Capitalize labels
actbind_cardioresp["cardiac"] = actbind_cardioresp["cardiac"].str.capitalize()
actbind_cardioresp["resp"] = actbind_cardioresp["resp"].str.capitalize()

### FIGURE 5a ###
# Raincloud plot of action binding according to cardio-respiratory phase
plot_cardioresp_binding(df_means_long=actbind_cardioresp,
                        yvar="act_binding",
                        ylabel="Action binding: OpA - BasA (ms)",
                        xlabel= "Respiratory phase at action onset", 
                        ylim=(-200,400),
                        palette=["#007A4E","#A4CBB6"],
                        svg_fname="Figure5a_ecg_resp_phase_actbinding.svg")


# Import dataset with tone binding means according to cardio-respiratory phase
tonebind_cardioresp = load_dataset(wd, 'cardioresp-phase-analysis', 
                                   f'task-{exp_name}_ecg_resp_tonebinding_avg.tsv')

# Capitalize labels
tonebind_cardioresp["cardiac"] = tonebind_cardioresp["cardiac"].str.capitalize()
tonebind_cardioresp["resp"] = tonebind_cardioresp["resp"].str.capitalize()

### FIGURE 5b ###
# Raincloud plot of tone binding according to cardio-respiratory phase
plot_cardioresp_binding(df_means_long=tonebind_cardioresp,
                        yvar="tone_binding",
                        ylabel="Tone binding: OpT - BasT (ms)",
                        xlabel= "Respiratory phase at tone onset", 
                        ylim=(-400,200),
                        palette=['#0B3142', '#7796CB'],
                        svg_fname="Figure5b_ecg_resp_phase_tonebinding.svg")